In [1]:
documents = [
    "Personal loan applicants must have a minimum credit score of 700.",
    
    "Home loan applicants require salary slips for the last 3 months.",
    
    "Loan applications above ₹20 lakh require manual risk assessment.",
    
    "Freelancers must provide 2 years of income tax returns.",
    
    "The maximum debt-to-income ratio allowed is 45%."
]

In [6]:
pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeableRequirement already satisfied: sentence-transformers in /usr/local/lib/python3.10/site-packages (2.2.2)Requirement already satisfied: transformers<5.0.0,>=4.6.0 in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (4.36.2)Requirement already satisfied: tqdm in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (4.67.1)Requirement already satisfied: torch>=1.6.0 in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (2.8.0)Requirement already satisfied: torchvision in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (0.23.0)Requirement already satisfied: numpy in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (1.26.4)Requirement already satisfied: scikit-learn in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (1.7.1)Requirement already satisfied: scipy in /usr/local/lib/python3.10/site-packages 

In [7]:
pip install sentence-transformers faiss-cpu openai numpy

Defaulting to user installation because normal site-packages is not writeableRequirement already satisfied: sentence-transformers in /usr/local/lib/python3.10/site-packages (2.2.2)Requirement already satisfied: faiss-cpu in /usr/local/lib/python3.10/site-packages (1.7.4)Requirement already satisfied: openai in /usr/local/lib/python3.10/site-packages (1.100.2)Requirement already satisfied: numpy in /usr/local/lib/python3.10/site-packages (1.26.4)Requirement already satisfied: transformers<5.0.0,>=4.6.0 in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (4.36.2)Requirement already satisfied: tqdm in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (4.67.1)Requirement already satisfied: torch>=1.6.0 in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (2.8.0)Requirement already satisfied: torchvision in /usr/local/lib/python3.10/site-packages (from sentence-transformers) (0.23.0)Requirement already satisfied: scikit-learn in /usr

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from openai import OpenAI
import os

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.10/site-packages/huggingface_hub/__init__.py)

In [0]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [0]:
documents = [
    "Personal loan applicants must have a minimum credit score of 700.",
    "Home loan applicants require salary slips for the last 3 months.",
    "Loan applications above ₹20 lakh require manual risk assessment.",
    "Freelancers must provide 2 years of income tax returns.",
    "The maximum debt-to-income ratio allowed is 45%."
]

doc_embeddings = embedding_model.encode(documents)

In [0]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [0]:
def retrieve_documents(query, top_k=2):
    
    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_docs = [documents[i] for i in indices[0]]

    return retrieved_docs

In [0]:
client = OpenAI(api_key=os.getenv("voc-1222044512175350470816369f4bca05d9987.79993304"))

In [0]:
def generate_rag_response(user_query):

    retrieved_docs = retrieve_documents(user_query)

    context = "\n".join(retrieved_docs)

    prompt = f"""
You are a Retail Banking Loan Advisory Assistant.

Use ONLY the retrieved banking policies below to answer.

Retrieved Policies:
{context}

Rules:
- Do not approve loans
- If information is missing, say "Need More Information"
- Explain risks clearly

User Query:
{user_query}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

In [0]:
def run_rag_agent():

    print("=== Loan Origination RAG Agent ===")

    while True:

        user_query = input("\nYou: ")

        if user_query.lower() == "exit":
            break

        retrieved_docs = retrieve_documents(user_query)

        print("\nRetrieved Documents:")
        for doc in retrieved_docs:
            print("-", doc)

        response = generate_rag_response(user_query)

        print("\nAgent Response:")
        print(response)